# H2S-NN Link Prediction Notebook

This notebook prepares the workspace and launches the H2S-NN link-prediction training entrypoint:

- `scripts/train_h2snn.py`
- `configs/experiments/lastfm_h2snn.yaml`
- `configs/experiments/amazonbook_h2snn.yaml`

Metrics reported: AUROC and AUPR (matching HetSheaf's link-prediction protocol).
Decoder is a swappable hyperparameter: `dot` (default), `distmult`, or `concat_mlp`.

Recommended runtime: GPU.

In [ ]:
from pathlib import Path
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/H2S-NN_test")
else:
    PROJECT_DIR = Path.cwd()

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"Project directory not found: {PROJECT_DIR}. Update PROJECT_DIR in this cell if needed."
    )

os.chdir(PROJECT_DIR)
print(f"Working directory: {PROJECT_DIR}")
print(f"Python executable: {sys.executable}")

In [ ]:
%pip install -r requirements.txt

In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Torch version: {torch.__version__}")
print(f"Device: {DEVICE}")
if DEVICE != "cuda":
    print("Warning: training will be much slower on CPU.")

In [ ]:
DATASET = "lastfm"  # one of: "lastfm", "amazonbook"
DECODER_TYPE = "dot"  # one of: "dot", "distmult", "concat_mlp"

CONFIG_BY_DATASET = {
    "lastfm": "configs/experiments/lastfm_h2snn.yaml",
    "amazonbook": "configs/experiments/amazonbook_h2snn.yaml",
}

CONFIG_PATH = CONFIG_BY_DATASET[DATASET]
EXP_NAME = f"{DATASET}_h2snn_notebook_{DECODER_TYPE}"
print(f"Config: {CONFIG_PATH}")
print(f"Decoder: {DECODER_TYPE}")
print(f"Experiment name: {EXP_NAME}")

In [ ]:
import os
import subprocess
import sys

command = [
    sys.executable,
    "-u",
    "scripts/train_h2snn.py",
    "--config",
    CONFIG_PATH,
    "--exp_name",
    EXP_NAME,
    "--device",
    DEVICE,
    "--decoder_type",
    DECODER_TYPE,
]

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

print("Running:", " ".join(command))
process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Training failed with exit code {return_code}")

In [ ]:
from pathlib import Path
import json

exp_dir = Path("experiments") / EXP_NAME
print("Experiment directory:", exp_dir)
print("Files:")
for path in sorted(exp_dir.rglob("*")):
    print(path)

metrics_path = exp_dir / "metrics.jsonl"
if metrics_path.exists():
    print("\nLast metric rows (val/test auroc, aupr):")
    lines = metrics_path.read_text(encoding="utf-8").strip().splitlines()
    for row in lines[-3:]:
        record = json.loads(row)
        print({
            "epoch": record["epoch"],
            "val_auroc": record["val"].get("auroc"),
            "val_aupr": record["val"].get("aupr"),
            "test_auroc": record["test"].get("auroc"),
            "test_aupr": record["test"].get("aupr"),
        })